# PT2: Writing a CuPy RawKernel for Adam/AdamW Paramaters

Since we already know how to write the Adam update step. Lets write a custom kernel that fuse the operations such that we perform multiple kernel passes in a single fused kernel, reducing the amount of trips to VRAM. 

We've already covered that the current version creates **~30 - 36** total kernels **PER TRAINABLE LAYER**, so we'll go over the simple code that covers how we'd write an optimized AdamW kernel that also accounts for Adam. At the end of the notebook, we'll see that the twokernels result in a sustained **11.5x speedup** on parameter updates compared to the vectorized baseline.

To actually write the new raw kernel, we'll actually write it the same exact way that we did for the pooling kernels. However, this version will be much simpler as we're dealing with only elementwise operations, meaning our thread indexing as well as grid indexing is trivial, and the literal operations themselves also remain trivial. 

In [ ]:
from string import Template

_ADAM_W_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
    float* __restrict__ param,
    const float* __restrict__ grad,
    float* __restrict__ m,
    float* __restrict__ v,
    const float lr,
    const float b1,
    const float b2,
    const float eps,
    const float bc1,
    const float bc2,
    const float weight_decay,
    const float l1_lambda,
    const float l2_lambda,
    const int N
    
) {
    // since its cheap, we can calculate thread x immediately
    int out_idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (out_idx >= N) return;
    float g = grad[out_idx];
    float p = param[out_idx];
    float m_val = m[out_idx];
    float v_val = v[out_idx];

    if (l1_lambda > 0.0f){
        g += l1_lambda * (p < 0.0f ? -1.0f : 1.0f);
    }
    if (l2_lambda > 0.0f){
        g += l2_lambda * p;
    }
    if (weight_decay > 0.0f) {
        p -= lr * weight_decay * p;
    }

    m_val = b1 * m_val + (1.0f - b1) * g;
    v_val = b2 * v_val + (1.0f - b2) * (g * g);

    float m_hat = m_val / bc1;
    float v_hat = v_val / bc2;

    p -= lr * m_hat / (sqrtf(v_hat) + eps);

    param[out_idx] = p;
    m[out_idx] = m_val;
    v[out_idx] = v_val;
}   
''')

### How does it work? 

Again, the operations behind the raw kernel itself are quite trivial. In essence, all that's happened is that we've computed the 1D thread offset inside `out_idx`, and then performed operations based on the thread index at that point. We can afford to perform if statements per thread, as these take around 2 clock cycles to perform, and we fetch the tensors into cache once we do the checks.
```c++
    float g = grad[out_idx];
    float p = param[out_idx];
    float m_val = m[out_idx];
    float v_val = v[out_idx];
```

Finally, we write back our three tensors, `layer.weights`, `layer.param_moment`, and `layer.param_cache`.

After writing the kernel, we'll still memoize the kernel based on the vendor type, and pass in the appropriate parameters inside the kernel. 

* `is_gpu_adamw_available`: We'll call `_get_compiled_adamw_kernel` to check for the compiled kernel.

* `_get_compiled_adamw_kernel`: We'll compile the raw kernel for the given vender, replacing the `$hip_include` subsitution with nothing or the hip import based on if the user is using AMD hardware or not. Next, we'll build the kernel once and afterwards cache the lookup for the kernel for subsequent accesses, bringing down overhead from compiling the same kernel over and over again. 

* `launch_adamw_update`: Finally, this is the kernel we directly call inside `Optimizer._step_gpu`. In this case, we pass in the kwargs that are required to compute regularization, weight decay, beta hyperparams, bias corrections, and the thensors to then perform an update on `layer.params` (bias and or weights).
Full stack is below:

In [1]:
from string import Template
import numpy as np
import aether.config as config

_ADAMW_CUDA = dict(hip_include="")
_ADAMW_HIP = dict(hip_include="#include <hip/hip_runtime.h>\n")

_adamw_kernel_cache = {}


def _get_compiled_adamw_kernel(variant: str):
    """Cached, compiled fused AdamW RawKernel for the given vendor variant.

    Single kernel signature 'variant' ('cuda' or 'hip') is the only cache axis,
    the same memoization pattern as pooling_kernel.py's
    _get_compiled_max_backward_kernel. A failed compile is cached as None so
    a broken build doesn't re-attempt (and re-warn) on every optimizer step.
    """
    if variant in _adamw_kernel_cache:
        return _adamw_kernel_cache[variant]

    vendor = _ADAMW_HIP if variant == "hip" else _ADAMW_CUDA
    kernel_name = f"fused_adamw_update_{variant}_kernel"
    source = _ADAM_W_TEMPLATE.substitute(
        kernel_name=kernel_name,
        **vendor,
    )

    kernel = config.build_kernel(
        lambda: config.cp.RawKernel(source, kernel_name),
        name=f"fused_adamw_update_{variant}",
    )
    _adamw_kernel_cache[variant] = kernel
    return kernel


def is_gpu_adamw_available(variant: str) -> bool:
    """Checks if CuPy hardware support and the fused kernel are loaded/compiled."""
    return _get_compiled_adamw_kernel(variant) is not None


def launch_adamw_update(
    kernel,
    param, grad, momentum, cache,
    lr, beta1, beta2, eps,
    bias_correction1, bias_correction2,
    weight_decay, l1_reg, l2_reg,
    block_size: int = 512,
):
    """Launches the fused AdamW update kernel over a flat parameter buffer.

    Assumes `param`/`grad`/`momentum`/`cache` are C-contiguous float32 arrays
    of identical shape -- true by construction at every allocation site in
    this project (Dense/Conv2d weight init)
    """
    size = param.size
    blocks = (size + block_size - 1) // block_size

    kernel(
        (blocks,), (block_size,),
        (
            param, grad, momentum, cache,
            np.float32(lr), np.float32(beta1), np.float32(beta2), np.float32(eps),
            np.float32(bias_correction1), np.float32(bias_correction2),
            np.float32(weight_decay), np.float32(l1_reg), np.float32(l2_reg),
            np.int32(size),
        ),
    )

### Performance Gain over Base Vectorized AdamW

Below builds a simple **MLP** that includes the optimizer **AdamW**. 

1. **Vectorized CuPy Fallback**: We'll create the network as normal, but right before we train, we can manually switch the backend to use the vectorized path instead writing `model.optimizer._compile_for_device("numpy")`; this allows us to still test the exact same model on the same hardware (GPU) and keeps the comparison like for like

2. **Fused CUDA RawKernel**: This won't overwrite the actual fast path, but it's still equivalent to manually writing `model.optimizer._compile_for_device("numpy")`, just that this is handled inside `model.to()` for us. 

In [5]:
import time
import cupy as cp
import numpy as np
import aether as ae

def build_mlp_model(input_shape=(32, 32, 3)):
    model = ae.Model()
    model.add(ae.Flatten())
    model.add(ae.Dense(32 * 32 * 3, 256))
    model.add(ae.ReLU())
    model.add(ae.Dense(256, 128, l2=1e-5))
    model.add(ae.Dense(128, 10))
    
    model.configure(
        loss=ae.SoftmaxCategoricalCrossEntropy(label_smoothing=0.01),
        optimizer=ae.AdamW(learning_rate=0.001, decay=5e-5, weight_decay=0.01),
        accuracy=ae.CategoricalAccuracy()
    )
    
    # Push weights and buffers to CuPy
    model.to('cupy')
    model.set_precision(compute_dtype="float16")
    model.finalize(input_shape=input_shape)
    
    # Populate dummy gradients on all trainable layers for benchmarking
    for layer in getattr(model, "trainable_layers", []):
        layer.dweights = cp.random.randn(*layer.weights.shape, dtype=layer.weights.dtype)
        if layer.biases is not None:
            layer.dbiases = cp.random.randn(*layer.biases.shape, dtype=layer.biases.dtype)
            
    return model

# 1. Base Model: Vectorized fallback path on GPU arrays
model_base = build_mlp_model()
model_base.optimizer._compile_for_device('numpy')  # Forces vectorized un-fused path

# 2. Optimized Model: Fused CUDA RawKernel path
model_fused = build_mlp_model()

def benchmark_optimizer_step(model, steps=2048):
    # Warmup pass
    for _ in range(10):
        model.optimizer.step()
    cp.cuda.Stream.null.synchronize()

    # Timed run
    start = time.perf_counter()
    for _ in range(steps):
        model.optimizer.step()
    cp.cuda.Stream.null.synchronize()
    end = time.perf_counter()

    return (end - start) / steps * 1000.0  # ms per step

time_base = benchmark_optimizer_step(model_base)
time_fused = benchmark_optimizer_step(model_fused)

print(f"Vectorized CuPy Fallback : {time_base:.4f} ms / step")
print(f"Fused CUDA RawKernel     : {time_fused:.4f} ms / step")
print(f"Speedup Factor           : {time_base / time_fused:.2f}x")

Vectorized CuPy Fallback : 0.3716 ms / step
Fused CUDA RawKernel     : 0.0322 ms / step
Speedup Factor           : 11.54x


### Results

Saturating the number of blocks needed to compute the updates show we achieve $\approx 11.50\text{x}$ speedup over the baseline fallback path. Even though this kernel is strictly memory bound, the fact we reduce the number of kernel launches from $30$ to $36$ kernel launches to $2$ still demonstrates the power of combining operations. 